In [2]:
import pandas as pd 
books=pd.read_csv("books_with_categories.csv")

In [6]:
#sentiment analysis as text classification
from transformers import pipeline
classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None, device=0)
classifier(books["description"][0].split("."))


Device set to use cpu


[[{'label': 'surprise', 'score': 0.7296027541160583},
  {'label': 'neutral', 'score': 0.14038576185703278},
  {'label': 'fear', 'score': 0.06816209107637405},
  {'label': 'joy', 'score': 0.04794240742921829},
  {'label': 'anger', 'score': 0.00915635284036398},
  {'label': 'disgust', 'score': 0.0026284719351679087},
  {'label': 'sadness', 'score': 0.0021221607457846403}],
 [{'label': 'neutral', 'score': 0.449370414018631},
  {'label': 'disgust', 'score': 0.27359241247177124},
  {'label': 'joy', 'score': 0.10908260941505432},
  {'label': 'sadness', 'score': 0.09362703561782837},
  {'label': 'anger', 'score': 0.040478307753801346},
  {'label': 'surprise', 'score': 0.02697017230093479},
  {'label': 'fear', 'score': 0.006879065651446581}],
 [{'label': 'neutral', 'score': 0.6462168097496033},
  {'label': 'sadness', 'score': 0.2427326887845993},
  {'label': 'disgust', 'score': 0.04342268034815788},
  {'label': 'surprise', 'score': 0.028300466015934944},
  {'label': 'joy', 'score': 0.014211482

In [7]:
#extract the maximum emotion probability for each emotion for each description
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

def calculate_max_emotion_scores(predictions):
    """Extract max emotion scores from classifier predictions."""
    per_emotion_scores = {label: [] for label in emotion_labels}
    
    for prediction in predictions:
        for pred in prediction:
            label = pred['label']
            score = pred['score']
            if label in per_emotion_scores:
                per_emotion_scores[label].append(score)
    
    return {label: np.max(scores) if scores else 0 for label, scores in per_emotion_scores.items()}

In [8]:
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [9]:
emotion_scores

{'anger': [np.float64(0.06413363665342331),
  np.float64(0.6126185059547424),
  np.float64(0.06413363665342331),
  np.float64(0.3514834940433502),
  np.float64(0.0814124047756195),
  np.float64(0.23222540318965912),
  np.float64(0.5381839871406555),
  np.float64(0.06413363665342331),
  np.float64(0.30066895484924316),
  np.float64(0.06413363665342331)],
 'disgust': [np.float64(0.27359241247177124),
  np.float64(0.3482835590839386),
  np.float64(0.10400672256946564),
  np.float64(0.15072275698184967),
  np.float64(0.18449535965919495),
  np.float64(0.7271743416786194),
  np.float64(0.1558549404144287),
  np.float64(0.10400672256946564),
  np.float64(0.27948135137557983),
  np.float64(0.17792679369449615)],
 'fear': [np.float64(0.928168535232544),
  np.float64(0.942527711391449),
  np.float64(0.9723208546638489),
  np.float64(0.36070650815963745),
  np.float64(0.0950433686375618),
  np.float64(0.05136279761791229),
  np.float64(0.7474278807640076),
  np.float64(0.40449780225753784),
  np

In [10]:
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5197/5197 [19:34<00:00,  4.43it/s] 


In [ ]:
emotions_df=pd.DataFrame(emotion_scores)
emotions_df["isbn13"]=isbn
books=pd.merge(books,emotions_df, on="isbn13")
books.to_csv("books_with_emotions.csv", index=False)